# Seminário 3: Adição de novas interações

## Autores

| Nome                                      | nUSP     |
| :---------------------------------------- | :------- |
| Lucas de Oliveira Ferreira                | 13695042 |
| Guilherme de Abreu Barreto                | 12543033 |
| Jhonathan Oliveira Alves                  | 11838116 |
| Lucas Pereira Franco de Almeida           | 12675020 |
| Miguel Prates Ferreira de Lima Cantanhede | 13672745 |


## Descrição

Neste notebook descrevemos o acrécimo de duas funcionalidades às visualizações as quais mencionamos no seminário anterior. À saber:

1. Integramos as viusalizações do Diagrama de Arcos e Gráfico de Linhas em um único painel, tal que as seleções feitas no menu de dropdown ou no gráfico de arcos sejam refletidas em ambas as visualizações (prática esta conhecida como _Linkng_);
2. Acrescentamos o zoom semântico à visualização do Gráfico de Bolhas, afim de facilitar a navegação neste.

In [1]:
from sqlalchemy import create_engine
import pandas as pd


engine = create_engine("sqlite:///database/lattes.db") # Acesso ao banco de dados

## Integração entre o Diagrama de Arcos e o Gráfico de Linhas

Pra melhor visualização, recomenda-se o acesso em uma [nova aba](http://127.0.0.1:8060/).

In [1]:
# Inicia o app combinado (Diagrama de Arcos + Gráfico de Linhas sincronizados)
from combined_dashboard import create_combined_dashboard
from line_graph.data import get_data
import sqlite3, pandas as pd

# carrega dados necessários
df, df_counts = get_data()
conn = sqlite3.connect('database/lattes.db')
collaborations_query = """
WITH article_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        a.title AS collaboration,
        'artigo' AS type,
        a.year AS start,
        a.year AS end
    FROM authorship au1
    JOIN authorship au2 ON au1.article_id = au2.article_id AND au1.author_id < au2.author_id
    JOIN researchers r1 ON au1.author_id = r1.lattes_id
    JOIN researchers r2 ON au2.author_id = r2.lattes_id
    JOIN articles a ON au1.article_id = a.id
),
project_collaborations AS (
    SELECT
        r1.name AS researcher_1,
        r2.name AS researcher_2,
        p.name AS collaboration,
        'projeto' AS type,
        p.start AS start,
        CAST(COALESCE(p.end, strftime('%Y', 'now')) AS INTEGER) AS end
    FROM participation p1
    JOIN participation p2 ON p1.project_id = p2.project_id AND p1.participant_id < p2.participant_id
    JOIN researchers r1 ON p1.participant_id = r1.lattes_id
    JOIN researchers r2 ON p2.participant_id = r2.lattes_id
    JOIN projects p ON p1.project_id = p.id
)
SELECT * FROM article_collaborations
UNION ALL
SELECT * FROM project_collaborations
"""
collaborations = pd.read_sql_query(collaborations_query, conn)

app = create_combined_dashboard(collaborations, df, df_counts)
PORT = 8060
HOST = 'localhost'
print(f"Starting combined dashboard at {HOST}:{PORT}")
app.run(host=HOST, port=PORT)

Starting combined dashboard at localhost:8060


Servidor rodando em http://localhost:8000/bubble_plot/fisheye.html


127.0.0.1 - - [24/Nov/2025 19:35:15] code 404, message File not found
127.0.0.1 - - [24/Nov/2025 19:35:15] "GET /bubble_plot/fisheye.html HTTP/1.1" 404 -
127.0.0.1 - - [24/Nov/2025 19:35:16] code 404, message File not found
127.0.0.1 - - [24/Nov/2025 19:35:16] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [24/Nov/2025 19:36:18] "GET /bubble_plot/fisheye.html HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2025 19:36:18] "GET /bubble_plot/bubbles.json HTTP/1.1" 200 -


## Bubble plot com zoom semântico

Pra melhor visualização, recomenda-se o acesso em uma [nova aba](http://127.0.0.1:8060/).

In [3]:
import os
import threading
import webbrowser
from http.server import HTTPServer, SimpleHTTPRequestHandler

PORT = 8000

def run_server():
    httpd = HTTPServer(("localhost", PORT), SimpleHTTPRequestHandler)
    print(f"Servidor rodando em http://localhost:{PORT}/bubble_plot/fisheye.html")
    httpd.serve_forever()

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

webbrowser.open(f"http://localhost:{PORT}/bubble_plot/fisheye.html")


Exception in thread Thread-61 (run_server):
Traceback (most recent call last):
  File "/nix/store/xgin1zcm9nx2bim7x8dlmn72kfasmnag-python3-3.13.7-env/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/home/user/Public/usp/computer-science-course/semester-6/computer-visualization/.devenv/state/venv/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/nix/store/xgin1zcm9nx2bim7x8dlmn72kfasmnag-python3-3.13.7-env/lib/python3.13/threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/run/user/1000/ipykernel_285589/687961729.py", line 9, in run_server
    httpd = HTTPServer(("localhost", PORT), SimpleHTTPRequestHandler)
  File "/nix/store/xgin1zcm9nx2bim7x8dlmn72kfasmnag-python3-3.13.7-env/lib/python3.13/socketserver.py", line 457, in __init__
    self.server_bind()
    

True